# Deliverable 2 - Primera solución

## Problema identificado en Deliverable 1

Durante el Deliverable 1, se observó que el uso de un modelo de lenguaje de escala pequeña mediante un único prompt directo (Baseline) presenta debilidades críticas al procesar transcripciones extensas y complejas:
1. **Confusión de propuestas con decisiones finales**: Mezcla debates iniciales o alternativas descartadas como si fueran acuerdos definitivos.
2. **Pérdida de decisiones reemplazadas**: No logra mapear la evolución temporal de un acuerdo cuando una decisión posterior anula a una anterior.
3. **Fusión de tareas distintas**: Agrupa múltiples entregables independientes en una única tarea ambigua.
4. **Asignación incorrecta de responsables e invención de plazos**: Asocia compromisos a participantes que no los adquirieron o inventa fechas límite sin sustento en el texto.
5. **Inestabilidad de formato**: Riesgo latente de producir JSON malformado debido a la extensión y la exigencia del esquema.

## Solución propuesta: Descomposición en Dos Etapas (Pipeline P1 + P2)

Para mitigar estos fallos, implementamos un Pipeline de procesamiento descompuesto en dos fases:
- **Fase P1 - Extracción Estructurada**: Se especializa únicamente en barrer el texto original para recopilar de forma estricta los candidatos a decisiones, tareas y asuntos pendientes, ligando siempre cada elemento a su evidencia textual exacta y su respectivo `utterance_id`. No asume fechas absolutas ni toma decisiones finales de estado.
- **Fase P2 - Resolución y Acta Final**: Recibe tanto la transcripción como los candidatos estructurados de P1. Su objetivo es consolidar el acta final resolviendo plazos relativos en función de la fecha de la reunión, depurando estados finales (`final`, `superseded`, `rejected`), validando responsables reales y descartando alucinaciones.

## Controles Experimentales Mantenidos

Para asegurar la validez científica y comparativa, mantenemos los siguientes parámetros estrictamente idénticos para el Baseline y el Pipeline:
* **Modelo**: `ministral-3:3b` (ejecutado localmente vía Ollama)
* **Tamaño del Contexto**: 8192 tokens
* **Temperatura**: 0 (determinismo máximo)
* **Semilla (Seed)**: 42
* **Límite de salida (num_predict)**: 3000 tokens
* **Formato**: JSON obligatorio

## Persistencia y Reanudación ante Desconexiones

Dado que el entorno de ejecución de Google Colab es efímero y puede perderse en sesiones largas, el flujo implementa las siguientes salvaguardas:
1. **Montaje de Google Drive**: Todo resultado final se almacena directamente en la ruta persistente `/content/drive/MyDrive/GenAI_VIO`.
2. **Reanudación por archivo (Checkpointing)**: Antes de procesar cada reunión (M01-M10), el script verifica si ya existe su archivo `*_resultado_completo.json` en Drive. Si ya fue procesada, la omite y avanza rápidamente, evitando costos innecesarios de tiempo y procesamiento.

## Segunda intervención: Decodificación restringida por esquema (v2)

En la primera versión, P1 y P2 usaban `format: "json"`, que solo garantiza que la salida sea JSON válido, pero no qué campos ni qué valores contiene. En la v2 cada etapa recibe un **esquema JSON construido a partir de la propia transcripción**, y Ollama impide que el modelo genere cualquier token que no calce con ese esquema:

| Falla del Deliverable 1 | Restricción del esquema | Métrica que debería mejorar |
|---|---|---|
| M3 atribución por saliencia | `assignee` y `owner` solo aceptan participantes presentes de esa reunión, o `null` | responsables correctos; asignaciones a no participantes = 0 |
| M4 sin abstención (IDs inventados) | `utterance_id` solo acepta IDs que existen en la transcripción; `candidate_id` sigue un formato fijo | evidencia válida |
| M1 colapso de modalidad | P1 debe etiquetar cada candidato con `kind` de una lista cerrada (`proposal`, `agreement`, `rejection`, `replacement`, `pending`) | recall de decisiones; estados correctos |
| M2 sin seguimiento de estado | P1 enlaza cambios con `replaces_candidate_ids`; `state` solo acepta `final`, `superseded`, `rejected`; P2 usa `kind`, los reemplazos y el orden de los IDs para decidir | estados correctos |
| Inestabilidad de formato | toda la salida sigue el esquema; `deadline` con formato de fecha fijo | JSON válido; plazos correctos |

El **baseline no cambia** (sigue con `format: "json"` y el prompt directo), para que la comparación sea contra el mismo punto de partida del Deliverable 1. La corrida anterior del pipeline sin esquema se conserva como condición intermedia (ablación).

## Versiones evaluadas (ablación)

*Ablación* = comparar el sistema agregando o quitando piezas de a una, para saber cuánto aporta cada una. Todas las versiones usan el mismo modelo, los mismos parámetros y las mismas 10 reuniones.

| Versión | Qué cambia | Por qué (diagnóstico que la motivó) |
|---|---|---|
| Baseline | Prompt directo en una llamada | Punto de partida del Deliverable 1 |
| v1 | P1 + P2 con `format: "json"` | Descomposición en etapas (M1, M2) |
| v2 | v1 + decodificación restringida por esquema, y el texto del esquema agregado al prompt de P2 | Valores cerrados para responsables, IDs y estados (M3, M4) |
| v3 | v2 + regla 7 reescrita ("no omitir") y sin el texto del esquema en P2 | En v2, P2 podaba decisiones reemplazadas/rechazadas y se saturaba el contexto en M01 y M02 |
| **v4 (final)** | v2 sin el texto del esquema en P2 (regla 7 original) | En v3, la regla nueva vació el contenido de 12 de 29 decisiones; v4 aísla el efecto de liberar contexto |

**Criterio estricto de decisiones:** una decisión solo cuenta como recuperada si su campo `outcome` dice qué se decidió. Si está vacío o solo repite el estado ("final", "superseded"), no cuenta, porque según la definición del Deliverable 1 cada campo debe estar respaldado y el acta debe registrar lo acordado. El criterio se aplica igual a todas las versiones.

## Paso 1: Configurar Entorno y Verificar Archivos en Drive

Montamos Google Drive y definimos las rutas de trabajo. Buscamos el archivo comprimido `GenAI_VIO.zip` en la carpeta permanente de Drive y lo extraemos en el almacenamiento temporal local para acelerar la lectura de archivos.

In [ ]:
import os, shutil, subprocess, time, requests

MODEL = 'ministral-3:3b'
HOST = 'http://127.0.0.1:11434'
LOG_PATH = '/content/ollama.log'

def run(cmd):
    return subprocess.run(cmd, shell=isinstance(cmd, str), check=True)

def ollama_ready():
    try:
        return requests.get(f'{HOST}/api/tags', timeout=3).ok
    except requests.RequestException:
        return False

if shutil.which('ollama') is None:
    print('Preparando dependencias e instalando Ollama...')
    run('apt-get update -qq && apt-get install -y -qq zstd')
    run('curl -fsSL https://ollama.com/install.sh | sh')

if not ollama_ready():
    print('Iniciando servicio Ollama...')
    with open(LOG_PATH, 'ab') as log:
        subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if ollama_ready():
            break
        time.sleep(1)
    else:
        raise RuntimeError(f'Ollama no inició. Revisa {LOG_PATH}')

installed = {m['name'] for m in requests.get(f'{HOST}/api/tags', timeout=10).json().get('models', [])}
if MODEL not in installed and f'{MODEL}:latest' not in installed:
    print(f'Descargando modelo {MODEL}...')
    run(['ollama', 'pull', MODEL])

print('✅ Ollama listo. Modelos:', [m['name'] for m in requests.get(f'{HOST}/api/tags', timeout=10).json()['models']])

Preparando dependencias e instalando Ollama...
Iniciando servicio Ollama...
Descargando modelo ministral-3:3b...
✅ Ollama listo. Modelos: ['ministral-3:3b']


In [ ]:
import requests
r = requests.post('http://127.0.0.1:11434/api/generate', json={'model':'ministral-3:3b','prompt':'Responde exclusivamente con el JSON {\"estado\":\"ok\"}.','stream':False,'format':'json','options':{'temperature':0,'num_predict':20}}, timeout=180)
r.raise_for_status()
print('✅ Generación verificada:', r.json()['response'])

✅ Generación verificada: {"estado":"ok"}


In [ ]:
import zipfile
from pathlib import Path
from google.colab import drive

# 1. Montar Google Drive
drive.mount("/content/drive")

# 2. Definir rutas de persistencia usando el nombre detectado por Drive
DRIVE_ROOT = Path("/content/drive/MyDrive/GenAI_VIO")
ZIP_PATH = DRIVE_ROOT / "GenAI_VIO.zip"

# Carpeta de ejecución local temporal
ROOT = Path("/content/genai-vio")

# 3. Validar existencia de ZIP
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"⛔ No se encontró 'GenAI_VIO.zip' en {DRIVE_ROOT}. "
        f"Por favor, crea la carpeta en tu Google Drive y sube el archivo ZIP manualmente antes de continuar."
    )
else:
    print(f"✔ ZIP detectado en: {ZIP_PATH}")

# 4. Descomprimir en ruta local
if not ROOT.exists():
    ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(ROOT)

print(f"✔ Proyecto descomprimido exitosamente en: {ROOT}")

Mounted at /content/drive
✔ ZIP detectado en: /content/drive/MyDrive/GenAI_VIO/GenAI_VIO.zip
✔ Proyecto descomprimido exitosamente en: /content/genai-vio


## Paso 2: Autodetectar Estructura del Repositorio

El contenido del archivo ZIP puede estar organizado directamente en la raíz o anidado dentro de una carpeta como `Fine Tuning` o similar. Corremos una autodetección inteligente de directorios para asegurar que los scripts lean las transcripciones, las rúbricas de evaluación y los prompts desde el lugar correcto.

In [ ]:
# Buscar recursivamente las carpetas clave dentro de ROOT
def autodetect_subfolders(root_dir):
    possible_transcripts = list(root_dir.glob("**/Transcripciones"))
    possible_rubrics = list(root_dir.glob("**/Rubricas"))
    possible_prompts = list(root_dir.glob("**/prompts"))
    possible_tests = list(root_dir.glob("**/pruebas"))

    if not possible_transcripts:
        raise FileNotFoundError("No se pudo encontrar la carpeta 'Transcripciones' dentro del ZIP descomprimido.")

    transcripts_path = possible_transcripts[0]
    rubrics_path = possible_rubrics[0] if possible_rubrics else None
    prompts_path = possible_prompts[0] if possible_prompts else None
    tests_path = possible_tests[0] if possible_tests else None

    return transcripts_path, rubrics_path, prompts_path, tests_path

TRANSCRIPTS_DIR, RUBRICS_DIR, PROMPTS_DIR, TESTS_DIR = autodetect_subfolders(ROOT)

print("Rutas de trabajo detectadas:")
print(f"  • Transcripciones: {TRANSCRIPTS_DIR}")
print(f"  • Rúbricas: {RUBRICS_DIR}")
print(f"  • Prompts (P1): {PROMPTS_DIR}")
print(f"  • Pruebas (Baseline): {TESTS_DIR}")

Rutas de trabajo detectadas:
  • Transcripciones: /content/genai-vio/Transcripciones
  • Rúbricas: /content/genai-vio/Rubricas
  • Prompts (P1): /content/genai-vio/prompts
  • Pruebas (Baseline): /content/genai-vio/pruebas


## Paso 3: Definición del Prompt P2 (Resolución y Consolidación)

Definimos en español las instrucciones estrictas de la Etapa P2. Este prompt se encarga de cruzar los candidatos del JSON de P1 con la transcripción original, resolver las fechas relativas en función del campo `meeting.date` de la reunión, estructurar los estados de decisión y estructurar el entregable final.

In [ ]:
RESOLUTION_PROMPT = """
Eres la etapa final de un sistema de actas de reunión verificables y estructuradas (Etapa P2 - Resolución).
Recibirás dos fuentes de información:
1. El JSON generado por la Etapa P1 con candidatos a decisiones, tareas y asuntos pendientes.
2. La transcripción completa original de la reunión.

Tu trabajo consiste en cruzar ambas fuentes para resolver, validar y emitir el acta final consolidada en formato JSON, sin omitir información clave ni inventar datos.

Sigue estas reglas de negocio de manera estricta:
1. **Estados de Decisiones**: Clasifica las decisiones identificadas en:
   - "final": Acuerdo definitivo y vigente alcanzado al cierre.
   - "superseded": Acuerdo que fue reemplazado o modificado por otra decisión posterior en la misma reunión.
   - "rejected": Propuesta discutida que fue explícitamente rechazada.
2. **Asuntos no resueltos**: Si un tema quedó pendiente, se postergó o no se tomó una decisión definitiva, colócalo bajo la lista de `pending_issues`.
3. **Resolución de Plazos y Fechas**: Traduce cualquier fecha relativa (ej. "mañana", "este viernes", "en dos semanas") a una fecha absoluta en formato 'YYYY-MM-DD' o 'YYYY-MM-DD HH:MM', calculándola estrictamente a partir de la fecha de la reunión dada en el campo `meeting.date`.
4. **Asignación de Responsables**: Asigna responsables a las tareas únicamente si están explícitamente respaldados en el texto. No asumas ni inventes personas, herramientas o acuerdos.
5. **Verificación de Evidencia Textual**: Cada elemento de decisión, tarea o pendiente debe incluir su respectiva evidencia textual. El `utterance_id` debe existir en la transcripción y la cita (`exact_quote`) debe estar presente de manera idéntica y literal en el texto.
6. **Valores Nulos**: Si falta un dato opcional (como responsable, plazos o condiciones), usa obligatoriamente el valor JSON `null` real. Jamás utilices la cadena de texto "null".
7. **Uso de los candidatos de P1**: P1 entrega `decision_candidates` con un `kind` y sus `utterance_id` (los IDs están en orden cronológico). Un candidato `proposal` sin un `agreement` posterior sobre el mismo tema NO es decisión final. Si un candidato aparece en `replaces_candidate_ids` de otro, el reemplazado es "superseded" y el que lo reemplaza es "final" (salvo que a su vez sea reemplazado). Un `rejection` es "rejected". Un `pending` va a `pending_issues`. Las `task_candidates` y `pending_issue_candidates` de P1 son la base de `action_items` y `pending_issues`.
8. **Plazos desde P1**: P1 entrega `deadline_expression` tal como se dijo (por ejemplo "el martes antes de la una"). Conviértela a fecha absoluta usando `meeting.date`; si la expresión es ambigua, usa null y agrega una alerta en `review_alerts`.
9. **Responsables**: El esquema solo permite participantes presentes o null. Si la persona que realiza la tarea no está entre los participantes, usa null.
10. **Formato Exclusivo**: Devuelve estrictamente el objeto JSON que se adecúe al esquema provisto. No incluyas explicaciones, introducciones ni bloques de Markdown (como ```json) en tu respuesta final.

Esquema de salida JSON requerido:
{
  "meeting": {
    "date": "YYYY-MM-DD",
    "topic": "string"
  },
  "decisions": [
    {
      "topic": "string",
      "outcome": "string",
      "state": "final | superseded | rejected",
      "evidence": [
        {
          "utterance_id": "string",
          "exact_quote": "string"
        }
      ]
    }
  ],
  "action_items": [
    {
      "task": "string",
      "assignee": "string | null",
      "deadline": "YYYY-MM-DD HH:MM | YYYY-MM-DD | null",
      "conditions": "string | null",
      "status": "agreed | conditional | pending",
      "evidence": [
        {
          "field": "task | assignee | deadline | conditions | status",
          "utterance_id": "string",
          "exact_quote": "string"
        }
      ]
    }
  ],
  "pending_issues": [
    {
      "issue": "string",
      "owner": "string | null",
      "deadline": "YYYY-MM-DD HH:MM | YYYY-MM-DD | null",
      "evidence": [
        {
          "utterance_id": "string",
          "exact_quote": "string"
        }
      ]
    }
  ],
  "review_alerts": [
    {
      "alert": "string",
      "reason": "string",
      "utterance_ids": [
        "string"
      ]
    }
  ]
}
"""

## Paso 4: Definición de Clases de Apoyo, Evaluación y Cliente de Ollama Robustecido

Implementamos una llamada de generación robusta (`call_ollama`) capaz de detectar respuestas JSON inválidas y realizar exactamente un reintento de reparación enviando un prompt corrector al modelo. Asimismo, programamos la lógica de evaluación automática que contrasta las respuestas con las rúbricas patrón (sin que el modelo acceda jamás a ellas).

In [ ]:
import csv
import json
import re
import time
import unicodedata
from datetime import datetime, timezone
from difflib import SequenceMatcher
import requests

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

def normalize_text(val):
    if val is None:
        return ""
    val = unicodedata.normalize("NFD", str(val).lower())
    val = "".join(c for c in val if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9]+", " ", val).strip()

def get_text_similarity(a, b):
    a, b = normalize_text(a), normalize_text(b)
    if not a or not b:
        return 0.0
    tokens_a, tokens_b = set(a.split()), set(b.split())
    jaccard = len(tokens_a & tokens_b) / len(tokens_a | tokens_b) if (tokens_a | tokens_b) else 0.0
    seq_ratio = SequenceMatcher(None, a, b).ratio()
    return max(jaccard, seq_ratio)

def parse_safely(text):
    try:
        # Limpiar posibles delimitadores de markdown si el modelo falló e incluyó ```json
        clean_text = text.strip()
        if clean_text.startswith("```"):
            lines = clean_text.splitlines()
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip() == "```":
                lines = lines[:-1]
            clean_text = "\n".join(lines).strip()
        return json.loads(clean_text), True
    except Exception:
        return None, False

NUM_CTX = 8192
NUM_PREDICT = 3000

def call_ollama(prompt, label, system_instruction="", fmt="json"):
    """fmt puede ser "json" (solo JSON válido) o un dict con un esquema JSON
    (decodificación restringida: el modelo no puede salirse del esquema)."""
    payload = {
        "model": "ministral-3:3b",
        "prompt": prompt,
        "stream": False,
        "format": fmt,
        "keep_alive": "10m",
        "options": {
            "num_ctx": NUM_CTX,
            "temperature": 0,
            "seed": 42,
            "num_predict": NUM_PREDICT,
        },
    }
    if system_instruction:
        payload["system"] = system_instruction

    started = time.perf_counter()
    res = requests.post(OLLAMA_URL, json=payload, timeout=1800)
    if not res.ok:
        # Con esquemas, un error aquí suele significar que Ollama no aceptó alguna regla del esquema
        raise RuntimeError(f"Ollama devolvió {res.status_code} en '{label}': {res.text[:500]}")
    elapsed = time.perf_counter() - started

    data = res.json()
    raw_response = data.get("response", "")
    parsed, ok = parse_safely(raw_response)
    retried = False
    original_raw = ""

    # Control de contexto: Ollama recorta en silencio el inicio del prompt si no cabe
    prompt_tokens = data.get("prompt_eval_count") or 0
    out_tokens = data.get("eval_count") or 0
    context_warning = None
    if prompt_tokens >= 0.95 * NUM_CTX:
        context_warning = f"posible recorte del prompt ({prompt_tokens}/{NUM_CTX} tokens)"
    elif prompt_tokens + out_tokens > NUM_CTX:
        context_warning = f"prompt+salida exceden el contexto ({prompt_tokens}+{out_tokens}>{NUM_CTX})"
    if data.get("done_reason") == "length":
        context_warning = (context_warning or "") + " | salida cortada por num_predict"
    if context_warning:
        print(f"   [!] {label}: {context_warning}")

    # Reintento de reparación si no es un JSON válido
    if not ok:
        retried = True
        original_raw = raw_response
        print(f"   [!] JSON inválido en '{label}'. Solicitando reparación...")
        repair_prompt = f"{raw_response}\n\nDevuelve exclusivamente una versión JSON válida del contenido anterior. No agregues explicación ni información nueva."

        payload["prompt"] = repair_prompt
        repair_start = time.perf_counter()
        res_rep = requests.post(OLLAMA_URL, json=payload, timeout=900)
        repair_elapsed = time.perf_counter() - repair_start
        elapsed += repair_elapsed

        data_rep = res_rep.json()
        raw_response = data_rep.get("response", "")
        parsed, ok = parse_safely(raw_response)

    return {
        "label": label,
        "wall_seconds": round(elapsed, 3),
        "response_json_valid": ok,
        "response_raw": raw_response,
        "response_original_unrepaired": original_raw if retried else None,
        "response": parsed,
        "retried_repair": retried,
        "done_reason": data.get("done_reason"),
        "prompt_eval_count": prompt_tokens,
        "eval_count": out_tokens,
        "context_warning": context_warning,
        "total_duration_ns": data.get("total_duration"),
        "prompt_eval_duration_ns": data.get("prompt_eval_duration"),
        "eval_duration_ns": data.get("eval_duration"),
    }

def check_evidence_validity(item, utterance_map):
    evidence = item.get("evidence", [])
    if not evidence:
        return False
    for ev in evidence:
        uid = ev.get("utterance_id")
        quote = ev.get("exact_quote", "")
        original = utterance_map.get(uid, "")
        if quote and normalize_text(quote) in normalize_text(original):
            return True
    return False

def match_items_greedy(gold_list, pred_list, gold_key_fn, pred_key_fn, threshold=0.4):
    matches = []
    for gi, gold in enumerate(gold_list):
        for pi, pred in enumerate(pred_list):
            sim = get_text_similarity(gold_key_fn(gold), pred_key_fn(pred))
            if sim >= threshold:
                matches.append((sim, gi, pi))

    matches.sort(reverse=True, key=lambda x: x[0])
    final_matches = []
    matched_g = set()
    matched_p = set()
    for score, gi, pi in matches:
        if gi not in matched_g and pi not in matched_p:
            matched_g.add(gi)
            matched_p.add(pi)
            final_matches.append((gi, pi, score))
    return final_matches, matched_g, matched_p

def extract_decisions_from_p2(resp):
    resp = resp or {}
    items = []
    for d in resp.get("decisions", []):
        items.append({"text": f"{d.get('topic', '')} {d.get('outcome', '')}", "state": d.get("state"), "raw": d})
    for p in resp.get("pending_issues", []):
        items.append({"text": p.get("issue", ""), "state": "pending", "raw": p})
    return items

def extract_tasks_from_p2(resp):
    resp = resp or {}
    return [{
        "text": t.get("task", ""),
        "assignee": t.get("assignee"),
        "deadline": t.get("deadline"),
        "conditions": t.get("conditions"),
        "raw": t
    } for t in resp.get("action_items", [])]

def normalizar_fecha(valor):
    if valor in (None, "", "null"):
        return None
    return str(valor).strip().replace(" ", "T")


def evaluate_pipeline(response, rubric, transcript):
    if not response:
        return {"json_valid": False}

    utterance_map = {u["id"]: u["text"] for u in transcript["utterances"]}
    participants = set(transcript.get("participants", []))

    gold_decisions = rubric.get("decisions", [])
    gold_tasks = rubric.get("tasks", [])

    pred_decisions = extract_decisions_from_p2(response)
    pred_tasks = extract_tasks_from_p2(response)

    dec_matches, matched_g_d, matched_p_d = match_items_greedy(
        gold_decisions,
        pred_decisions,
        lambda x: f"{x.get('topic', '')} {x.get('statement', '')}",
        lambda x: x["text"]
    )

    task_matches, matched_g_t, matched_p_t = match_items_greedy(
        gold_tasks,
        pred_tasks,
        lambda x: x.get("description", ""),
        lambda x: x["text"],
        threshold=0.35
    )

    dec_states_ok = 0
    dec_ev_ok = 0
    for gi, pi, _ in dec_matches:
        g_state = gold_decisions[gi].get("state")
        p_state = pred_decisions[pi]["state"]
        if g_state == p_state:
            dec_states_ok += 1
        if check_evidence_validity(pred_decisions[pi]["raw"], utterance_map):
            dec_ev_ok += 1

    assignee_ok = 0
    deadline_ok = 0
    condition_ok = 0
    task_ev_ok = 0
    for gi, pi, _ in task_matches:
        gold = gold_tasks[gi]
        pred = pred_tasks[pi]

        if gold.get("assignee") == pred["assignee"]:
            assignee_ok += 1
        if normalizar_fecha(gold.get("deadline")) == normalizar_fecha(pred["deadline"]):
            deadline_ok += 1

        g_cond = gold.get("condition")
        p_cond = pred["conditions"]
        if g_cond is None and p_cond is None:
            condition_ok += 1
        elif g_cond and p_cond and get_text_similarity(g_cond, p_cond) >= 0.55:
            condition_ok += 1

        if check_evidence_validity(pred["raw"], utterance_map):
            task_ev_ok += 1

    invalid_assignees = sorted({
        t.get("assignee") for t in pred_tasks
        if t.get("assignee") not in (None, "", "null") and t.get("assignee") not in participants
    })

    return {
        "json_valid": True,
        "expected_decisions": len(gold_decisions),
        "found_decisions": len(dec_matches),
        "decision_recall": round(len(dec_matches) / max(1, len(gold_decisions)), 3),
        "correct_decision_states": dec_states_ok,
        "valid_decision_evidence": dec_ev_ok,
        "expected_tasks": len(gold_tasks),
        "found_tasks": len(task_matches),
        "task_recall": round(len(task_matches) / max(1, len(gold_tasks)), 3),
        "correct_assignees": assignee_ok,
        "correct_deadlines": deadline_ok,
        "correct_conditions": condition_ok,
        "valid_task_evidence": task_ev_ok,
        "invalid_assignees": invalid_assignees,
        "missing_decisions": [
            f"{x.get('topic')}: {x.get('statement')}" for i, x in enumerate(gold_decisions) if i not in matched_g_d
        ],
        "missing_tasks": [
            x.get("description") for i, x in enumerate(gold_tasks) if i not in matched_g_t
        ]
    }

## Paso 4b: Esquemas para la decodificación restringida

Construimos un esquema distinto para cada reunión, porque los valores permitidos salen de la propia transcripción:
- `utterance_id` solo puede ser un ID que exista en esa reunión.
- `assignee` / `owner` solo pueden ser participantes presentes de esa reunión, o `null`.
- `state`, `status`, `speech_act` y `field` solo aceptan valores de una lista cerrada.

`additionalProperties: false` impide que el modelo agregue campos que no están en el esquema, y `required` obliga a que estén todos.

In [ ]:
USE_PATTERNS = True   # si la celda de prueba falla por "pattern", cámbialo a False

P1_KINDS = ["proposal", "agreement", "rejection", "replacement", "pending"]  # los mismos del prompt P1

def _nullable(schema):
    return {"anyOf": [schema, {"type": "null"}]}

def _obj(props, required=None):
    return {"type": "object", "properties": props,
            "required": required or list(props.keys()), "additionalProperties": False}

def _deadline_schema():
    s = {"type": "string"}
    if USE_PATTERNS:
        s["pattern"] = "^[0-9]{4}-[0-9]{2}-[0-9]{2}( [0-9]{2}:[0-9]{2})?$"
    return _nullable(s)

def _ids_and_people(transcript):
    ids = [u["id"] for u in transcript["utterances"]]
    people = list(transcript.get("participants", []))
    return ids, people

def _id_schema(prefix):
    s = {"type": "string"}
    if USE_PATTERNS:
        s["pattern"] = f"^{prefix}[0-9]+$"
    return s

def build_p1_schema(transcript):
    """P1: mismo esquema que describe prompts/02_extraccion_descompuesta_ministral.md,
    pero con valores cerrados tomados de la propia reunión."""
    ids, people = _ids_and_people(transcript)
    evidence = _obj({"utterance_id": {"type": "string", "enum": ids},
                     "exact_quote": {"type": "string"}})
    task_evidence = _obj({"field": {"type": "string", "enum": ["task", "assignee", "deadline_expression", "conditions"]},
                          "utterance_id": {"type": "string", "enum": ids},
                          "exact_quote": {"type": "string"}})
    return _obj({
        "meeting": _obj({"date": {"type": "string", "enum": [transcript["meeting_date"]]},
                         "topic": _nullable({"type": "string"}),
                         "participants": {"type": "array", "items": {"type": "string", "enum": people}}}),
        "decision_candidates": {"type": "array", "items": _obj({
            "candidate_id": _id_schema("d"),
            "topic": {"type": "string"},
            "statement": {"type": "string"},
            "kind": {"type": "string", "enum": P1_KINDS},
            "replaces_candidate_ids": {"type": "array", "items": _id_schema("d")},
            "conditions": _nullable({"type": "string"}),
            "evidence": {"type": "array", "items": evidence, "minItems": 1},
        })},
        "task_candidates": {"type": "array", "items": _obj({
            "candidate_id": _id_schema("t"),
            "task": {"type": "string"},
            "assignee": _nullable({"type": "string", "enum": people}),
            "deadline_expression": _nullable({"type": "string"}),
            "conditions": _nullable({"type": "string"}),
            "evidence": {"type": "array", "items": task_evidence, "minItems": 1},
        })},
        "pending_issue_candidates": {"type": "array", "items": _obj({
            "issue": {"type": "string"},
            "owner": _nullable({"type": "string", "enum": people}),
            "deadline_expression": _nullable({"type": "string"}),
            "evidence": {"type": "array", "items": evidence, "minItems": 1},
        })},
    })

def build_p2_schema(transcript):
    """P2: acta final, mismo esquema del prompt directo pero con valores cerrados."""
    ids, people = _ids_and_people(transcript)
    evidence = _obj({"utterance_id": {"type": "string", "enum": ids},
                     "exact_quote": {"type": "string"}})
    task_evidence = _obj({"field": {"type": "string", "enum": ["task", "assignee", "deadline", "conditions", "status"]},
                          "utterance_id": {"type": "string", "enum": ids},
                          "exact_quote": {"type": "string"}})
    return _obj({
        "meeting": _obj({"date": {"type": "string", "enum": [transcript["meeting_date"]]},
                         "topic": {"type": "string"}}),
        "decisions": {"type": "array", "items": _obj({
            "topic": {"type": "string"},
            "outcome": {"type": "string"},
            "state": {"type": "string", "enum": ["final", "superseded", "rejected"]},
            "evidence": {"type": "array", "items": evidence, "minItems": 1},
        })},
        "action_items": {"type": "array", "items": _obj({
            "task": {"type": "string"},
            "assignee": _nullable({"type": "string", "enum": people}),
            "deadline": _deadline_schema(),
            "conditions": _nullable({"type": "string"}),
            "status": {"type": "string", "enum": ["agreed", "conditional", "pending"]},
            "evidence": {"type": "array", "items": task_evidence, "minItems": 1},
        })},
        "pending_issues": {"type": "array", "items": _obj({
            "issue": {"type": "string"},
            "owner": _nullable({"type": "string", "enum": people}),
            "deadline": _deadline_schema(),
            "evidence": {"type": "array", "items": evidence, "minItems": 1},
        })},
        "review_alerts": {"type": "array", "items": _obj({
            "alert": {"type": "string"},
            "reason": {"type": "string"},
            "utterance_ids": {"type": "array", "items": {"type": "string", "enum": ids}},
        })},
    })

def schema_instruction(schema):
    """Texto que se agrega al prompt: Ollama recomienda que el modelo también vea el esquema."""
    return ("\n\nFORMATO DE SALIDA OBLIGATORIO. Responde únicamente con un JSON que cumpla este esquema. "
            "Los valores permitidos en listas 'enum' son los únicos válidos:\n"
            + json.dumps(schema, ensure_ascii=False))

print("✔ Constructores de esquema listos")


✔ Constructores de esquema listos


## Paso 4c: Prueba de que la restricción funciona

Antes de correr las 10 reuniones, comprobamos con una reunión real que Ollama acepta los esquemas y que el modelo **no puede** asignar una tarea a alguien que no está en la lista de participantes, aunque se lo pidamos directamente.

In [ ]:
_t = json.loads(sorted(TRANSCRIPTS_DIR.glob("*_transcript.json"))[0].read_text(encoding="utf-8"))
_ids, _people = _ids_and_people(_t)
print("Participantes permitidos:", _people)

# 1) Los esquemas de P1 y P2 son aceptados por Ollama
_p1 = call_ollama("Devuelve una extracción vacía de esta reunión.", "prueba_esquema_p1", fmt=build_p1_schema(_t))
print("Esquema P1 aceptado, JSON válido:", _p1["response_json_valid"])
_mini = call_ollama(
    "Devuelve un acta vacía de esta reunión." + schema_instruction(build_p2_schema(_t)),
    "prueba_esquema_p2", fmt=build_p2_schema(_t))
print("Esquema P2 aceptado, JSON válido:", _mini["response_json_valid"])

# 2) El modelo no puede usar un responsable fuera de la lista
_probe_schema = _obj({"task": {"type": "string"},
                      "assignee": _nullable({"type": "string", "enum": _people})})
_probe = call_ollama(
    'La tarea "enviar el informe" quedó asignada a Paula Gómez. '
    'Devuelve {"task": ..., "assignee": ...}.',
    "prueba_enum_assignee", fmt=_probe_schema)
print("Respuesta a la trampa:", _probe["response"])
assert _probe["response"]["assignee"] in _people + [None], "⛔ La restricción NO se aplicó"
print("✔ La restricción funciona: el modelo no pudo escribir un nombre fuera de la lista")


Participantes permitidos: ['Speaker 01', 'Speaker 02', 'Speaker 03', 'Speaker 04']
Esquema P1 aceptado, JSON válido: True
Esquema P2 aceptado, JSON válido: True
Respuesta a la trampa: {'task': 'enviar el informe', 'assignee': 'Speaker 01'}
✔ La restricción funciona: el modelo no pudo escribir un nombre fuera de la lista


In [ ]:
t = json.loads(sorted(TRANSCRIPTS_DIR.glob("*_transcript.json"))[0].read_text(encoding="utf-8"))
r = json.loads(sorted(RUBRICS_DIR.glob("*_rubric.json"))[0].read_text(encoding="utf-8"))
print("Claves transcript:", list(t.keys()))
print("Participantes:", t.get("participants"))
print("Primera intervención:", t["utterances"][0])
print("Plazos en la rúbrica:", [x.get("deadline") for x in r["tasks"]])

Claves transcript: ['meeting_id', 'meeting_date', 'title', 'participants', 'utterances']
Participantes: ['Speaker 01', 'Speaker 02', 'Speaker 03', 'Speaker 04']
Primera intervención: {'id': 'u01', 'timestamp': '00:00:04', 'speaker': 'Speaker 01', 'text': 'Buenas tardes. La idea de hoy es cerrar fecha, formato y responsables para el piloto con clientes.'}
Plazos en la rúbrica: ['2026-09-14T18:00', '2026-09-14', None, '2026-09-24', '2026-09-18T17:00', '2026-09-14T17:00', '2026-09-14']


## Paso 5: Ejecución del Experimento Reproducible (M01-M10)

Iniciamos el ciclo experimental. El sistema autodetecta las reuniones, aplica la lógica de reanudación leyendo si ya se encuentra el resultado almacenado en el Drive para evitar reprocesos innecesarios, y guarda los resultados incrementalmente tras finalizar cada transcripción.

In [ ]:
import time
import subprocess
import requests
from datetime import datetime
import json
import csv
from pathlib import Path

# ---- Garantizar que Ollama está activo con reintentos y lectura de logs ----
def asegurar_ollama():
    print("🔍 Verificando estado del servicio de Ollama...")
    log_path = Path("/content/ollama.log")
    for i in range(10):
        try:
            res = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
            if res.status_code == 200:
                print("✔ Ollama está activo y respondiendo correctamente.")
                return True
        except requests.exceptions.ConnectionError:
            print(f"   [!] Ollama no responde en el intento {i+1}/10. Intentando iniciar servicio...")
            subprocess.run("nohup ollama serve > /content/ollama.log 2>&1 &", shell=True)
            time.sleep(4)

    # Si falla, imprimimos el log para saber qué ocurre
    print("\n⛔ Falló la conexión. Analizando logs de /content/ollama.log para diagnóstico:")
    if log_path.exists():
        print("=" * 50)
        print(log_path.read_text())
        print("=" * 50)
    else:
        print("No se generó ningún archivo de log en /content/ollama.log")

    raise ConnectionError("⛔ No se pudo establecer conexión con Ollama tras varios intentos. Revisa los detalles impresos arriba.")

asegurar_ollama()

# Directorio de salida persistente con nombre FIJO: así, si Colab se desconecta,
# al volver a correr la celda se reanuda donde quedó (con un nombre por fecha
# cada corrida creaba una carpeta nueva y nunca encontraba los checkpoints).
EXPERIMENT_NAME = "experimento_v4"
EXPERIMENT_DIR = DRIVE_ROOT / "resultados" / EXPERIMENT_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
OLLAMA_VERSION = subprocess.check_output(["ollama", "--version"], text=True).strip()
print("Versión de Ollama:", OLLAMA_VERSION, "| experimento:", EXPERIMENT_NAME)

# Cargar Prompts usando las variables de rutas dinámicas autodetectadas
BASELINE_PROMPT_PATH = TESTS_DIR / "01_prompt_directo_ministral.md"
EXTRACTION_PROMPT_PATH = PROMPTS_DIR / "02_extraccion_descompuesta_ministral.md"

assert BASELINE_PROMPT_PATH.exists(), f"No se encontró el prompt del baseline en {BASELINE_PROMPT_PATH}"
assert EXTRACTION_PROMPT_PATH.exists(), f"No se encontró el prompt de extracción (P1) en {EXTRACTION_PROMPT_PATH}"

BASELINE_INSTRUCTIONS = BASELINE_PROMPT_PATH.read_text(encoding="utf-8")
P1_INSTRUCTIONS = EXTRACTION_PROMPT_PATH.read_text(encoding="utf-8")

transcript_files = sorted(TRANSCRIPTS_DIR.glob("*_transcript.json"))

def count_invalid(ev):
    return len(ev.get("invalid_assignees", [])) if ev.get("json_valid") else None

def make_row(record):
    """Una fila del CSV a partir del registro guardado de una reunión."""
    eb, ep = record["evaluations"]["baseline"], record["evaluations"]["pipeline"]
    b, p = record["baseline"], record["pipeline"]
    return {
        "meeting_id": record["meeting_id"],
        "baseline_json_valido": b["response_json_valid"],
        "pipeline_json_valido": p["p2_json_valid"],
        "baseline_segundos": round(b["wall_seconds"], 2),
        "pipeline_segundos": round(p["total_wall_seconds"], 2),
        "baseline_recall_decisiones": eb.get("decision_recall", 0.0),
        "pipeline_recall_decisiones": ep.get("decision_recall", 0.0),
        "baseline_recall_tareas": eb.get("task_recall", 0.0),
        "pipeline_recall_tareas": ep.get("task_recall", 0.0),
        "baseline_plazos_correctos": eb.get("correct_deadlines", 0),
        "pipeline_plazos_correctos": ep.get("correct_deadlines", 0),
        "baseline_responsables_correctos": eb.get("correct_assignees", 0),
        "pipeline_responsables_correctos": ep.get("correct_assignees", 0),
        # columnas nuevas de la v2
        "decisiones_esperadas": ep.get("expected_decisions", eb.get("expected_decisions")),
        "tareas_esperadas": ep.get("expected_tasks", eb.get("expected_tasks")),
        "baseline_estados_correctos": eb.get("correct_decision_states", 0),
        "pipeline_estados_correctos": ep.get("correct_decision_states", 0),
        "baseline_asignados_no_participantes": count_invalid(eb),
        "pipeline_asignados_no_participantes": count_invalid(ep),
        "baseline_aviso_contexto": b.get("context_warning"),
        "pipeline_aviso_contexto": p.get("p1_context_warning") or p.get("p2_context_warning"),
    }

all_rows = []
human_findings = [
    "# Hallazgos automáticos de las pruebas (M01-M10)",
    "",
    "Este informe compara el baseline directo con un pipeline de dos etapas:",
    "- P1 extrae candidatos de forma estructurada.",
    "- P2 los unifica en un acta final resolviendo dependencias y plazos.",
    "Las rúbricas se mantuvieron secretas para el modelo, utilizándose únicamente para evaluación.",
    ""
]

for t_file in transcript_files:
    meeting_num = t_file.stem.split("_")[0]
    meeting_id = f"M{meeting_num}"

    # Comprobar si ya existe checkpoint en la carpeta del experimento actual
    checkpoint_file = EXPERIMENT_DIR / f"{meeting_id}_resultado_completo.json"
    if checkpoint_file.exists():
        print(f"⏩ {meeting_id} ya existe: se reutiliza el resultado guardado")
        all_rows.append(make_row(json.loads(checkpoint_file.read_text(encoding="utf-8"))))
        continue

    print(f"\n===== Ejecutando {meeting_id} =====")

    # Buscar rúbrica correspondiente
    rubric_file = RUBRICS_DIR / f"{meeting_num}_rubric.json"
    if not rubric_file.exists():
        print(f"⚠️ Rúbrica faltante para {meeting_id}. Se omite.")
        continue

    transcript_data = json.loads(t_file.read_text(encoding="utf-8"))
    rubric_data = json.loads(rubric_file.read_text(encoding="utf-8"))

    # ---- A. Ejecución del Baseline ----
    baseline_prompt = f"{BASELINE_INSTRUCTIONS}\n\n--- TRANSCRIPCIÓN A ANALIZAR ---\n\n{json.dumps(transcript_data, ensure_ascii=False, indent=2)}"
    print(f"   └─ Corriendo Baseline...")
    baseline_res = call_ollama(baseline_prompt, f"{meeting_id}_baseline")

    # ---- B. Ejecución del Pipeline P1 + P2 ----
    print(f"   └─ Corriendo Pipeline P1 (Extracción)...")
    p1_schema = build_p1_schema(transcript_data)
    # El prompt P1 ya describe este mismo esquema; aquí solo se agrega la restricción
    p1_prompt = f"{P1_INSTRUCTIONS}\n\n--- TRANSCRIPCIÓN A ANALIZAR ---\n\n{json.dumps(transcript_data, ensure_ascii=False, indent=2)}"
    p1_res = call_ollama(p1_prompt, f"{meeting_id}_P1_extraccion", fmt=p1_schema)

    if p1_res["response_json_valid"]:
        print(f"   └─ Corriendo Pipeline P2 (Resolución)...")
        p2_schema = build_p2_schema(transcript_data)
        # JSON compacto (sin sangría) para que candidatos + transcripción quepan en 8K
        compact = dict(ensure_ascii=False, separators=(",", ":"))
        # v4: sin el texto del esquema (la restricción la aplica Ollama vía format=p2_schema)
        p2_prompt = (f"{RESOLUTION_PROMPT}"
                     f"\n\n--- CANDIDATOS EXTRAÍDOS POR P1 ---\n\n{json.dumps(p1_res['response'], **compact)}"
                     f"\n\n--- TRANSCRIPCIÓN ORIGINAL ---\n\n{json.dumps(transcript_data, **compact)}")
        p2_res = call_ollama(p2_prompt, f"{meeting_id}_P2_resolucion", fmt=p2_schema)
    else:
        p2_res = {
            "label": f"{meeting_id}_P2_resolucion",
            "wall_seconds": 0.0,
            "response_json_valid": False,
            "response_raw": "",
            "response": None,
            "error": "P1 falló en generar un JSON válido."
        }

    # ---- Evaluaciones automáticas ----
    eval_baseline = evaluate_pipeline(baseline_res["response"], rubric_data, transcript_data)
    eval_pipeline = evaluate_pipeline(p2_res["response"], rubric_data, transcript_data)

    time_baseline = baseline_res["wall_seconds"]
    time_p1 = p1_res["wall_seconds"]
    time_p2 = p2_res["wall_seconds"]
    time_pipeline = time_p1 + time_p2

    # ---- Guardar el JSON del resultado completo por reunión ----
    record = {
        "meeting_id": meeting_id,
        "transcript": t_file.name,
        "rubric": rubric_file.name,
        "baseline": {
            "wall_seconds": time_baseline,
            "response_json_valid": baseline_res["response_json_valid"],
            "response": baseline_res["response"],
            "retried_repair": baseline_res.get("retried_repair", False),
            "context_warning": baseline_res.get("context_warning"),
            "prompt_eval_count": baseline_res.get("prompt_eval_count"),
        },
        "pipeline": {
            "p1_wall_seconds": time_p1,
            "p2_wall_seconds": time_p2,
            "total_wall_seconds": time_pipeline,
            "p1_json_valid": p1_res["response_json_valid"],
            "p2_json_valid": p2_res["response_json_valid"],
            "p1_response": p1_res["response"],
            "p2_response": p2_res["response"],
            "p1_retried_repair": p1_res.get("retried_repair", False),
            "p2_retried_repair": p2_res.get("retried_repair", False),
            "p1_context_warning": p1_res.get("context_warning"),
            "p2_context_warning": p2_res.get("context_warning"),
            "p1_prompt_eval_count": p1_res.get("prompt_eval_count"),
            "p2_prompt_eval_count": p2_res.get("prompt_eval_count"),
        },
        "variant": EXPERIMENT_NAME,
        "ollama_version": OLLAMA_VERSION,
        "evaluations": {
            "baseline": eval_baseline,
            "pipeline": eval_pipeline
        }
    }

    checkpoint_file.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"   ✔ Fichero de guardado incremental creado para {meeting_id}.")

    # ---- Construcción del Reporte Humano (hallazgos_simples.md) ----
    finding = f"## {meeting_id}\n"
    finding += f"- **Prueba realizada**: Comparativa directa de Baseline versus Pipeline de dos etapas (P1 + P2).\n"
    finding += f"- **Tiempo del Baseline**: {time_baseline:.2f} s | **Tiempo del Pipeline**: {time_pipeline:.2f} s (P1: {time_p1:.2f} s, P2: {time_p2:.2f} s).\n"

    if eval_baseline.get("json_valid"):
        finding += f"- **Resultados Baseline**: Recuperó {eval_baseline['found_decisions']}/{eval_baseline['expected_decisions']} decisiones y {eval_baseline['found_tasks']}/{eval_baseline['expected_tasks']} tareas.\n"
    else:
        finding += "- **Resultados Baseline**: Retornó un JSON inválido (no analizable automáticamente).\n"

    if eval_pipeline.get("json_valid"):
        finding += f"- **Resultados Pipeline**: Recuperó {eval_pipeline['found_decisions']}/{eval_pipeline['expected_decisions']} decisiones y {eval_pipeline['found_tasks']}/{eval_pipeline['expected_tasks']} tareas.\n"
        if eval_pipeline["missing_decisions"]:
            finding += f"  - *Decisiones omitidas*: {'; '.join(eval_pipeline['missing_decisions'])}\n"
        if eval_pipeline["missing_tasks"]:
            finding += f"  - *Tareas omitidas*: {'; '.join(eval_pipeline['missing_tasks'])}\n"
        if eval_pipeline["invalid_assignees"]:
            finding += f"  - *Alerta de responsables*: Se asignó trabajo a personas no participantes: {', '.join(eval_pipeline['invalid_assignees'])}.\n"
    else:
        finding += "- **Resultados Pipeline**: Retornó un JSON inválido en P1 o P2.\n"

    finding += "- *Casos para revisión manual*: Todas las métricas de coinciencia semántica son de carácter orientativo. Coincidencias o exclusiones dudosas requieren inspección manual directa en el archivo JSON mediante el utterance_id y exact_quote.\n"
    human_findings.append(finding)

    # ---- Recopilar métricas para resumen_metricas.csv ----
    all_rows.append(make_row(record))

# ---- Guardar los informes consolidados finales en Google Drive ----
csv_path = EXPERIMENT_DIR / "resumen_metricas.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    if all_rows:
        writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
        writer.writeheader()
        writer.writerows(all_rows)

findings_path = EXPERIMENT_DIR / "hallazgos_simples.md"
findings_path.write_text("\n".join(human_findings), encoding="utf-8")

print("\n" + "=" * 60)
print("✔ Experimento finalizado.")
print(f"📂 Resultados globales almacenados en Drive: {EXPERIMENT_DIR}")
print(f"📈 Métricas resumidas en: {csv_path.name}")
print(f"✍ Informe escrito en: {findings_path.name}")

🔍 Verificando estado del servicio de Ollama...
✔ Ollama está activo y respondiendo correctamente.
Versión de Ollama: ollama version is 0.34.4 | experimento: experimento_v4

===== Ejecutando M01 =====
   └─ Corriendo Baseline...
   └─ Corriendo Pipeline P1 (Extracción)...
   └─ Corriendo Pipeline P2 (Resolución)...
   ✔ Fichero de guardado incremental creado para M01.

===== Ejecutando M02 =====
   └─ Corriendo Baseline...
   └─ Corriendo Pipeline P1 (Extracción)...
   └─ Corriendo Pipeline P2 (Resolución)...
   ✔ Fichero de guardado incremental creado para M02.

===== Ejecutando M03 =====
   └─ Corriendo Baseline...
   └─ Corriendo Pipeline P1 (Extracción)...
   └─ Corriendo Pipeline P2 (Resolución)...
   ✔ Fichero de guardado incremental creado para M03.

===== Ejecutando M04 =====
   └─ Corriendo Baseline...
   └─ Corriendo Pipeline P1 (Extracción)...
   └─ Corriendo Pipeline P2 (Resolución)...
   ✔ Fichero de guardado incremental creado para M04.

===== Ejecutando M05 =====
   └─ Co

## Paso 6: Resumen Ejecutivo e Implicaciones para el Equipo

El siguiente script lee y consolida la tabla de métricas generada con el fin de arrojar un resumen claro sobre la efectividad del Baseline versus el Pipeline descompuesto, las limitaciones encontradas y las recomendaciones a seguir.

In [ ]:
import pandas as pd

metrics_df = pd.read_csv(csv_path)

# Estadísticos consolidados
total_meetings = len(metrics_df)
avg_time_base = metrics_df["baseline_segundos"].mean()
avg_time_pipe = metrics_df["pipeline_segundos"].mean()
mean_rec_dec_base = metrics_df["baseline_recall_decisiones"].mean() * 100
mean_rec_dec_pipe = metrics_df["pipeline_recall_decisiones"].mean() * 100
mean_rec_tar_base = metrics_df["baseline_recall_tareas"].mean() * 100
mean_rec_tar_pipe = metrics_df["pipeline_recall_tareas"].mean() * 100

print("============================================================")
print("         RESUMEN DE DESEMPEÑO EJECUTIVO DEL PROYECTO        ")
print("============================================================")
print(f"• Modelo utilizado: {MODEL}")
print(f"• Cantidad de reuniones analizadas: {total_meetings}")
print(f"• Ubicación de resultados: {EXPERIMENT_DIR}")
print("------------------------------------------------------------")
print("COMPARATIVA DE TIEMPOS DE EJECUCIÓN (PROMEDIO):")
print(f"  • Baseline: {avg_time_base:.2f} s")
print(f"  • Pipeline (P1+P2): {avg_time_pipe:.2f} s (Incremento del ~{((avg_time_pipe/avg_time_base)-1)*100:.1f}%)")
print("------------------------------------------------------------")
print("EFECTIVIDAD DE RECUPERACIÓN (RECALL PROMEDIO):")
print(f"  • Decisiones -> Baseline: {mean_rec_dec_base:.1f}% | Pipeline (P1+P2): {mean_rec_dec_pipe:.1f}%")
print(f"  • Tareas     -> Baseline: {mean_rec_tar_base:.1f}% | Pipeline (P1+P2): {mean_rec_tar_pipe:.1f}%")
print("============================================================")
print("\nLIMITACIÓN PRINCIPAL OBSERVADA:")
print("El pipeline de dos etapas incrementa la latencia global y, aunque reduce errores de formato ")
print("y falsas decisiones al basarse estrictamente en candidatos de P1, sigue dependiendo críticamente ")
print("de la exhaustividad de la extracción inicial. Si P1 omite un elemento, P2 carece de la iniciativa ")
print("para autogenerarlo o recuperarlo.")

print("\nPRÓXIMO PASO RECOMENDADO:")
print("Optimizar la etapa P1. Es fundamental refinar las instrucciones del prompt de extracción en ")
print("prompts/02_extraccion_descompuesta_ministral.md, agregando técnicas de Few-Shot o guiado estructurado ")
print("específico para tareas altamente dependientes, responsabilidades cruzadas y cadenas de decisiones reemplazadas.")

         RESUMEN DE DESEMPEÑO EJECUTIVO DEL PROYECTO        
• Modelo utilizado: ministral-3:3b
• Cantidad de reuniones analizadas: 10
• Ubicación de resultados: /content/drive/MyDrive/GenAI_VIO/resultados/experimento_v4
------------------------------------------------------------
COMPARATIVA DE TIEMPOS DE EJECUCIÓN (PROMEDIO):
  • Baseline: 21.39 s
  • Pipeline (P1+P2): 45.87 s (Incremento del ~114.4%)
------------------------------------------------------------
EFECTIVIDAD DE RECUPERACIÓN (RECALL PROMEDIO):
  • Decisiones -> Baseline: 54.7% | Pipeline (P1+P2): 56.0%
  • Tareas     -> Baseline: 84.9% | Pipeline (P1+P2): 78.7%

LIMITACIÓN PRINCIPAL OBSERVADA:
El pipeline de dos etapas incrementa la latencia global y, aunque reduce errores de formato 
y falsas decisiones al basarse estrictamente en candidatos de P1, sigue dependiendo críticamente 
de la exhaustividad de la extracción inicial. Si P1 omite un elemento, P2 carece de la iniciativa 
para autogenerarlo o recuperarlo.

PRÓXIMO

## Paso 7: Tabla final (cinco condiciones, mismo criterio para todas)

Recalcula todas las métricas desde los resultados guardados en Drive, con el mismo evaluador para el baseline y las cuatro versiones del pipeline. Incluye el criterio estricto de decisiones y la banda de ruido del baseline (el mismo baseline corrido en tres sesiones distintas): una diferencia menor que esa banda no se puede reclamar como mejora.

In [ ]:
import json
import pandas as pd

RES = DRIVE_ROOT / "resultados"
v1_dir = sorted(d for d in RES.glob("experimento_2*") if any(d.glob("M*_resultado_completo.json")))[-1]
ESTADOS = {"final", "superseded", "rejected", "pending", ""}
gold_key = lambda x: f"{x.get('topic','')} {x.get('statement','')}"
vacio = lambda raw: (raw.get("outcome", raw.get("issue")) or "").strip().lower() in ESTADOS

def medir(carpeta, clave):
    a = dict(dec_gold=0, dec=0, dec_cont=0, estados=0, vacias=0, t_gold=0, t=0, resp=0, plazos=0, no_part=0, seg=0.0, n=0)
    for f in sorted(carpeta.glob("M*_resultado_completo.json")):
        rec = json.loads(f.read_text(encoding="utf-8"))
        rub = json.loads((RUBRICS_DIR / rec["rubric"]).read_text(encoding="utf-8"))
        tr = json.loads((TRANSCRIPTS_DIR / rec["transcript"]).read_text(encoding="utf-8"))
        resp = rec["baseline"]["response"] if clave == "baseline" else rec["pipeline"]["p2_response"]
        ev = evaluate_pipeline(resp, rub, tr)
        preds = extract_decisions_from_p2(resp)
        gd = rub.get("decisions", [])
        a["dec_gold"] += len(gd)
        a["dec"] += ev.get("found_decisions", 0)
        a["dec_cont"] += len(match_items_greedy(gd, [p for p in preds if not vacio(p["raw"])], gold_key, lambda x: x["text"])[0])
        a["estados"] += ev.get("correct_decision_states", 0)
        a["vacias"] += sum(vacio(p["raw"]) for p in preds)
        a["t_gold"] += len(rub.get("tasks", []))
        a["t"] += ev.get("found_tasks", 0)
        a["resp"] += ev.get("correct_assignees", 0)
        a["plazos"] += ev.get("correct_deadlines", 0)
        a["no_part"] += len(ev.get("invalid_assignees", []))
        a["seg"] += rec["baseline"]["wall_seconds"] if clave == "baseline" else rec["pipeline"]["total_wall_seconds"]
        a["n"] += 1
    return {
        "Reuniones": a["n"],
        "Decisiones emparejadas": f"{a['dec']}/{a['dec_gold']}",
        "Decisiones con contenido": f"{a['dec_cont']}/{a['dec_gold']} ({a['dec_cont']/max(a['dec_gold'],1):.0%})",
        "Estados correctos": a["estados"],
        "Decisiones vacías": a["vacias"],
        "Tareas emparejadas": f"{a['t']}/{a['t_gold']}",
        "Responsables correctos": a["resp"],
        "Plazos correctos": a["plazos"],
        "Asignados a no participantes": a["no_part"],
        "Tiempo promedio (s)": round(a["seg"] / max(a["n"], 1), 1),
    }

condiciones = {
    "Baseline": (RES / "experimento_v4", "baseline"),
    "v1 json": (v1_dir, "pipeline"),
    "v2 esquema": (RES / "experimento_v2_esquema", "pipeline"),
    "v3 regla 7 nueva": (RES / "experimento_v3", "pipeline"),
    "v4 final": (RES / "experimento_v4", "pipeline"),
}
tabla = pd.DataFrame({k: medir(c, cl) for k, (c, cl) in condiciones.items()})
print(tabla.to_string())
tabla.to_csv(RES / "tabla_final.csv")
print("\n✔ Guardada en", RES / "tabla_final.csv")

print("\nRuido del baseline entre sesiones (mismo sistema, misma T4, distinto día):")
for k in ["experimento_v2_esquema", "experimento_v3", "experimento_v4"]:
    m = medir(RES / k, "baseline")
    print(f"  {k:<24} decisiones con contenido {m['Decisiones con contenido']:<14} responsables {m['Responsables correctos']:<4} plazos {m['Plazos correctos']}")


                                 Baseline      v1 json   v2 esquema v3 regla 7 nueva     v4 final
Reuniones                              10           10           10               10           10
Decisiones emparejadas              31/56        34/56        27/56            36/56        31/56
Decisiones con contenido      30/56 (54%)  17/56 (30%)  27/56 (48%)      26/56 (46%)  26/56 (46%)
Estados correctos                      22           20           22               25           18
Decisiones vacías                       1           19            0               12            5
Tareas emparejadas                  41/53        41/53        39/53            38/53        38/53
Responsables correctos                 34           31           31               27           31
Plazos correctos                       19           23           24               22           25
Asignados a no participantes            1            0            0                0            0
Tiempo promedio (s) 

## Paso 8: Diagnóstico de la versión final

Responde dónde se pierden las decisiones (¿P1 no las extrae o P2 las descarta?), qué tan precisos son los estados sobre lo encontrado, qué tipo de error hay en los responsables, y si hubo avisos de contexto.

In [ ]:
import json
import pandas as pd
from collections import Counter

def p1_items(p1):
    p1 = p1 or {}
    items = [{"text": f"{d.get('topic','')} {d.get('statement','')}"} for d in p1.get("decision_candidates", [])]
    items += [{"text": p.get("issue", "")} for p in p1.get("pending_issue_candidates", [])]
    return items

gold_dec_key = lambda x: f"{x.get('topic','')} {x.get('statement','')}"
diag = []

for f in sorted(EXPERIMENT_DIR.glob("M*_resultado_completo.json")):
    rec = json.loads(f.read_text(encoding="utf-8"))
    rub = json.loads((RUBRICS_DIR / rec["rubric"]).read_text(encoding="utf-8"))
    gd, gt = rub.get("decisions", []), rub.get("tasks", [])
    fila = {"reunion": rec["meeting_id"], "dec_gold": len(gd)}

    m_p1, _, _ = match_items_greedy(gd, p1_items(rec["pipeline"]["p1_response"]), gold_dec_key, lambda x: x["text"])
    fila["dec_en_P1"] = len(m_p1)

    for name, resp in [("base", rec["baseline"]["response"]), ("final", rec["pipeline"]["p2_response"])]:
        preds_d, preds_t = extract_decisions_from_p2(resp), extract_tasks_from_p2(resp)
        md_, _, _ = match_items_greedy(gd, preds_d, gold_dec_key, lambda x: x["text"])
        fila[f"dec_{name}"] = len(md_)
        fila[f"estado_ok_{name}"] = sum(1 for gi, pi, _ in md_ if gd[gi].get("state") == preds_d[pi]["state"])

        mt, _, _ = match_items_greedy(gt, preds_t, lambda x: x.get("description", ""), lambda x: x["text"], threshold=0.35)
        c = Counter()
        for gi, pi, _ in mt:
            g = gt[gi].get("assignee")
            p = preds_t[pi]["assignee"]
            p = None if p in ("", "null") else p
            if g == p:
                c["ok"] += 1
            elif g is None:
                c["asigno_sin_responsable"] += 1
            elif p is None:
                c["null_indebido"] += 1
            else:
                c["participante_equivocado"] += 1
        for k in ["ok", "participante_equivocado", "null_indebido", "asigno_sin_responsable"]:
            fila[f"resp_{k}_{name}"] = c[k]

    fila["aviso_base"] = rec["baseline"].get("context_warning")
    fila["aviso_P1"] = rec["pipeline"].get("p1_context_warning")
    fila["aviso_P2"] = rec["pipeline"].get("p2_context_warning")
    fila["tokens_P2"] = rec["pipeline"].get("p2_prompt_eval_count")
    diag.append(fila)

df = pd.DataFrame(diag)
df.to_csv(EXPERIMENT_DIR / f"diagnostico_{EXPERIMENT_NAME}.csv", index=False)
S = df.sum(numeric_only=True)

print("DÓNDE SE PIERDEN LAS DECISIONES")
print(f"  Gold: {S.dec_gold:.0f} | extraídas por P1: {S.dec_en_P1:.0f} | quedan en P2: {S.dec_final:.0f} | baseline: {S.dec_base:.0f}")
print("\nPRECISIÓN DE ESTADOS SOBRE LO EMPAREJADO")
print(f"  Baseline: {S.estado_ok_base:.0f}/{S.dec_base:.0f} = {S.estado_ok_base/max(S.dec_base,1):.0%}")
print(f"  Final:    {S.estado_ok_final:.0f}/{S.dec_final:.0f} = {S.estado_ok_final/max(S.dec_final,1):.0%}")
print("\nTIPO DE ERROR EN RESPONSABLES (tareas emparejadas)")
for k in ["ok", "participante_equivocado", "null_indebido", "asigno_sin_responsable"]:
    print(f"  {k:<26} baseline {S[f'resp_{k}_base']:.0f} | final {S[f'resp_{k}_final']:.0f}")
print("\nAVISOS DE CONTEXTO")
avisos = df[df[["aviso_base", "aviso_P1", "aviso_P2"]].notna().any(axis=1)]
print("  Ninguno" if avisos.empty else avisos[["reunion", "aviso_base", "aviso_P1", "aviso_P2", "tokens_P2"]].to_string(index=False))

df["perdidas_P1_a_P2"] = df.dec_en_P1 - df.dec_final
print("\nPOR REUNIÓN")
print(df[["reunion", "dec_gold", "dec_base", "dec_en_P1", "dec_final", "perdidas_P1_a_P2", "tokens_P2"]].to_string(index=False))

perdidas, lista = Counter(), []
for f in sorted(EXPERIMENT_DIR.glob("M*_resultado_completo.json")):
    rec = json.loads(f.read_text(encoding="utf-8"))
    rub = json.loads((RUBRICS_DIR / rec["rubric"]).read_text(encoding="utf-8"))
    gd = rub.get("decisions", [])
    m_p1, _, _ = match_items_greedy(gd, p1_items(rec["pipeline"]["p1_response"]), gold_dec_key, lambda x: x["text"])
    m_p2, _, _ = match_items_greedy(gd, extract_decisions_from_p2(rec["pipeline"]["p2_response"]), gold_dec_key, lambda x: x["text"])
    for gi in {g for g, _, _ in m_p1} - {g for g, _, _ in m_p2}:
        perdidas[gd[gi].get("state")] += 1
        lista.append(f"  {rec['meeting_id']} [{gd[gi].get('state')}] {gold_dec_key(gd[gi])}")
print("\nEstado en el gold de las decisiones que P1 extrajo y P2 descartó:", dict(perdidas))
print("\n".join(lista))


DÓNDE SE PIERDEN LAS DECISIONES
  Gold: 56 | extraídas por P1: 44 | quedan en P2: 31 | baseline: 31

PRECISIÓN DE ESTADOS SOBRE LO EMPAREJADO
  Baseline: 22/31 = 71%
  Final:    18/31 = 58%

TIPO DE ERROR EN RESPONSABLES (tareas emparejadas)
  ok                         baseline 34 | final 31
  participante_equivocado    baseline 7 | final 6
  null_indebido              baseline 0 | final 1
  asigno_sin_responsable     baseline 0 | final 0

AVISOS DE CONTEXTO
  Ninguno

POR REUNIÓN
reunion  dec_gold  dec_base  dec_en_P1  dec_final  perdidas_P1_a_P2  tokens_P2
    M01         7         4          5          3                 2       4985
    M02         6         5          6          3                 3       5476
    M03         6         3          4          4                 0       4813
    M04         5         2          5          3                 2       4130
    M05         5         2          2          2                 0       3844
    M06         5         3          4 

## Paso 9: Caso de falla, origen de los responsables equivocados

Prueba la hipótesis de atribución por saliencia (M3): en transcripciones anonimizadas, ¿el modelo asigna la tarea a quien habla en la cita, en vez de a quien la recibe? También distingue si el error nace en P1 o en P2.

In [ ]:
from collections import Counter

def cargar(mid, carpeta=None):
    carpeta = carpeta or EXPERIMENT_DIR
    rec = json.loads((carpeta / f"{mid}_resultado_completo.json").read_text(encoding="utf-8"))
    rub = json.loads((RUBRICS_DIR / rec["rubric"]).read_text(encoding="utf-8"))
    tr = json.loads((TRANSCRIPTS_DIR / rec["transcript"]).read_text(encoding="utf-8"))
    return rec, rub, tr

origen, ejemplos = Counter(), []
for f in sorted(EXPERIMENT_DIR.glob("M*_resultado_completo.json")):
    mid = json.loads(f.read_text(encoding="utf-8"))["meeting_id"]
    rec, rub, tr = cargar(mid)
    habla = {u["id"]: u["speaker"] for u in tr["utterances"]}
    gt = rub.get("tasks", [])
    p2t = extract_tasks_from_p2(rec["pipeline"]["p2_response"])
    p1t = [{"text": t.get("task", ""), "assignee": t.get("assignee")}
           for t in (rec["pipeline"]["p1_response"] or {}).get("task_candidates", [])]
    m2, _, _ = match_items_greedy(gt, p2t, lambda x: x.get("description", ""), lambda x: x["text"], threshold=0.35)
    m1 = {gi: pi for gi, pi, _ in match_items_greedy(gt, p1t, lambda x: x.get("description", ""), lambda x: x["text"], threshold=0.35)[0]}
    for gi, pi, _ in m2:
        g, p = gt[gi].get("assignee"), p2t[pi]["assignee"]
        if g is None or p is None or g == p:
            continue
        a1 = p1t[m1[gi]]["assignee"] if gi in m1 else "(no extraída)"
        hablantes = [habla.get(e.get("utterance_id")) for e in p2t[pi]["raw"].get("evidence", [])]
        if a1 == p:
            origen["el error ya venía de P1"] += 1
        elif a1 == g:
            origen["P2 cambió un responsable correcto de P1"] += 1
        else:
            origen["otro origen"] += 1
        origen["responsable predicho = quien habla en la cita"] += int(p in hablantes)
        origen["responsable correcto = quien habla en la cita"] += int(g in hablantes)
        ejemplos.append(f"  {mid} | {gt[gi]['description'][:55]} | gold {g} | P1 {a1} | P2 {p} | hablan: {hablantes}")

print("ORIGEN DE LOS RESPONSABLES EQUIVOCADOS")
for k, v in origen.items():
    print(f"  {k}: {v}")
print("\n".join(ejemplos))


ORIGEN DE LOS RESPONSABLES EQUIVOCADOS
  el error ya venía de P1: 6
  responsable predicho = quien habla en la cita: 6
  responsable correcto = quien habla en la cita: 0
  M04 | Corregir el límite de carga | gold Speaker 04 | P1 Speaker 01 | P2 Speaker 01 | hablan: ['Speaker 01']
  M05 | Preparar el módulo de herramientas | gold Speaker 02 | P1 Speaker 03 | P2 Speaker 03 | hablan: ['Speaker 03']
  M06 | Publicar el aviso de venta de escritorios | gold Speaker 04 | P1 Speaker 02 | P2 Speaker 02 | hablan: ['Speaker 02']
  M07 | Revisar la redacción final | gold Speaker 02 | P1 Speaker 04 | P2 Speaker 04 | hablan: ['Speaker 04']
  M10 | Comprar los premios digitales | gold Speaker 01 | P1 Speaker 03 | P2 Speaker 03 | hablan: ['Speaker 03']
  M10 | Preparar la sección de logros del equipo | gold Speaker 04 | P1 Speaker 03 | P2 Speaker 03 | hablan: ['Speaker 03']


## Paso 10: Inspección directa de actas (M01 y M07)

Muestra lado a lado el gold y el acta final del pipeline, para revisar a simple vista casos que el evaluador automático podría juzgar mal. Para ver la v3 (donde apareció el contenido vacío), cambia `CARPETA` por `DRIVE_ROOT / "resultados" / "experimento_v3"`.

In [ ]:
CARPETA = EXPERIMENT_DIR

for mid in ["M01", "M07"]:
    rec, rub, _ = cargar(mid, CARPETA)
    print(f"\n===== {mid} ({CARPETA.name}) =====\nGOLD:")
    for d in rub["decisions"]:
        print(f"  [{d['state']}] {d['topic']}: {d['statement']}")
    print("ACTA DEL PIPELINE:")
    r = rec["pipeline"]["p2_response"] or {}
    for d in r.get("decisions", []):
        print(f"  [{d.get('state')}] {d.get('topic')}: {d.get('outcome')}")
    for p in r.get("pending_issues", []):
        print(f"  [pending] {p.get('issue')}")
    print("TAREAS:")
    for t in r.get("action_items", []):
        print(f"  {t.get('task')} | {t.get('assignee')} | {t.get('deadline')}")



===== M01 (experimento_v4) =====
GOLD:
  [superseded] Fecha y hora del piloto: Realizar el piloto el lunes 21 a las 10:00
  [superseded] Fecha y hora del piloto: Realizar el piloto el lunes 21 a las 16:00
  [final] Fecha y hora del piloto: Realizar el piloto el martes 22 a las 10:00
  [superseded] Formato de la actividad: Demostración de veinte minutos seguida de encuesta
  [final] Formato de la actividad: Demostración de quince minutos, prueba guiada y encuesta
  [pending] Escenario de la prueba guiada: Elegir entre un escenario de feedback o negociación de plazo
  [final] Uso de cámara en el piloto: No mostrar la funcionalidad de cámara durante el piloto
ACTA DEL PIPELINE:
  [final] Fecha del piloto: final
  [final] Duración de la demostración: final
  [pending] Escenario de la prueba guiada
TAREAS:
  Preparar el correo de invitación | Speaker 02 | 2026-09-14 18:00
  Armar la lista de empresas contactadas | Speaker 03 | 2026-09-14
  Consolidar comentarios de la encuesta | Speaker 02